# 01 · Una tabla con números, categorías y ausencias — solución de referencia

Inspeccionar una tabla y ajustar imputación y one-hot solo con entrenamiento, aceptando una categoría nueva.

**Ideas que aparecen:** laboratorio 00 y lectura de archivos (cápsula 2). Puedes consultar las cápsulas
opcionales de `ruta/Puentes de entrada.md` cuando alguna te haga falta.

Datos **sintéticos educativos**, creados en este repositorio y dedicados a
CC0-1.0. No representan personas ni un rendimiento oficial IOAI.
Todo el ejercicio usa CPU y archivos locales; no requiere cuentas ni red.

Esta es una propuesta para comparar con tus ideas; no es la única solución posible.
Puedes recorrerlo en una o varias sesiones. No hay límite de juez ni
obligación de completar todos los experimentos para abrir el siguiente cuaderno.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

DATOS = Path("datos")
assert DATOS.is_dir(), "Abre el notebook desde su carpeta: deben existir datos/ e inicio.ipynb."
SEMILLA = 17


## Contrato de los datos
Cada fila es una estación ficticia independiente. `temperatura`,
`lluvia` y `zona` están disponibles al predecir; `alerta` es el
objetivo sintético. `id` identifica la fila y `particion` fija el
experimento: ninguna de esas dos es una feature.

Imputar aprende un valor de reemplazo. One-hot convierte cada
categoría en una columna indicadora. `ColumnTransformer` aplica
procedimientos distintos a columnas distintas y el `Pipeline`
los ajusta junto con el modelo, exclusivamente sobre train.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
df = pd.read_csv(DATOS / "estaciones.csv")
print(df.shape, df.dtypes, df.isna().sum(), sep="\n")
print(df.groupby("particion")["alerta"].value_counts())
columnas = ["temperatura", "lluvia", "zona"]
train = df[df.particion == "train"]
val = df[df.particion == "val"]
nuevo = df[df.particion == "transferencia"]
assert (len(train), len(val), len(nuevo)) == (160, 40, 40)


## Zona de experimentación
Un pipeline puede combinar numéricas → mediana y escalado;
`zona` → valor más frecuente y one-hot; luego logística. La figura
ayuda a relacionar esas decisiones con lo que hay en la tabla.
Con `handle_unknown="ignore"`, una categoría nueva se representa
   con ceros en sus indicadores, no con una categoría conocida inventada.
No llames `fit` sobre validación ni sobre la tabla completa.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

def construir_modelo():
    numericas = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    categorias = make_pipeline(SimpleImputer(strategy="most_frequent"),
                               OneHotEncoder(handle_unknown="ignore"))
    preparar = ColumnTransformer([
        ("numericas", numericas, ["temperatura", "lluvia"]),
        ("categorias", categorias, ["zona"]),
    ])
    return make_pipeline(preparar, LogisticRegression(max_iter=1000, random_state=SEMILLA))


## Comparación en las mismas condiciones
Accuracy sobre las mismas 40 filas de validación para baseline mayoritario y propuesta. La solución muestra una mejora sobre el baseline local; su valor no equivale a ningún score oficial. Observar transferencia con la propuesta fijada permite distinguir selección de evaluación.


In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(train[columnas], train.alerta)
modelo = construir_modelo().fit(train[columnas], train.alerta)
pred = modelo.predict(val[columnas])
resultado = {
    "baseline": float(accuracy_score(val.alerta, baseline.predict(val[columnas]))),
    "validacion": float(accuracy_score(val.alerta, pred)),
    "nulos": {c: int(df[c].isna().sum()) for c in columnas},
}
assert len(pred) == len(val) and set(pred) <= {0, 1}
print(resultado)


<details><summary>Idea · orden</summary>La columna particion ya fija el split; selecciona filas antes de ajustar el modelo.</details>
<details><summary>Idea · API</summary>Combina SimpleImputer, StandardScaler y OneHotEncoder mediante ColumnTransformer.</details>
<details><summary>Idea · novedad</summary>Comprueba el conjunto de categorías de train y luego las de transferencia. No reentrenes el encoder con ambas.</details>


## Transferencia: decide antes de mirar el resultado
En transferencia aparece `cordillera`, ausente de train. Congela la propuesta antes de ejecutar. Explica cómo la procesa y qué limitación tiene ignorar una categoría nueva. Revisa el score sin volver a ajustar hiperparámetros con estas etiquetas.


In [ ]:
categorias_nuevas = sorted(set(nuevo.zona.dropna()) - set(train.zona.dropna()))
pred_nuevo = modelo.predict(nuevo[columnas])
resultado["transferencia"] = float(accuracy_score(nuevo.alerta, pred_nuevo))
resultado["baseline_transferencia"] = float(accuracy_score(nuevo.alerta, baseline.predict(nuevo[columnas])))
resultado["categorias_nuevas"] = categorias_nuevas
assert categorias_nuevas == ["cordillera"] and len(pred_nuevo) == 40


## Mirar la idea
Compara el dibujo con lo que esperabas antes de ejecutar.


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].scatter(train.temperatura, train.lluvia, c=train.alerta, cmap="coolwarm", edgecolor="white")
ejes[0].set(title="Train: dos variables, una decisión", xlabel="temperatura", ylabel="lluvia")
ejes[1].bar(columnas, [train[c].isna().sum() for c in columnas], color="#2a7f9e")
ejes[1].set(title="Lo que falta también necesita una decisión", ylabel="ausencias en entrenamiento")
fig.tight_layout()
plt.show()
# Los puntos con una coordenada ausente no aparecen en el scatter; la barra los hace visibles.


## Para seguir explorando
¿Qué aprende cada transformación y de qué filas? Quita una feature o cambia una categoría por un nombre nuevo. ¿Falla el programa, cambia la predicción o conserva exactamente la misma? Usa el dibujo para buscar ejemplos donde una sola variable no basta.


## Comprobaciones del ejemplo
Estas aserciones detectan errores técnicos en el cuaderno y en su solución de referencia. No son un examen ni una escala de capacidad.


In [ ]:
assert resultado['validacion'] >= 0.75 and resultado['validacion'] > resultado['baseline']
preparar = modelo.steps[0][1]
assert np.allclose(preparar.named_transformers_['numericas'].steps[0][1].statistics_, train[['temperatura', 'lluvia']].median())


## Resultado reproducible
Esta celda guarda automáticamente las medidas para comprobar el material. Puedes conservar una copia de tu notebook y tus propias notas; no hay un formulario que rellenar.


In [ ]:
resultado.update({"laboratorio": '01_tabla', "version": 'solución de referencia',
                  "datos": "sintéticos CC0-1.0",
                  "metrica": 'accuracy (mayor es mejor)', "split": 'columna particion fija: 160 entrenamiento, 40 validación, 40 transferencia'})
Path("resultado.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=2))
